In [2]:
#pip install pandas openpyxl

In [1]:
import pandas as pd
import re

# Función para contar los registros que no cumplen la precisión
def contar_precision(df, columna, columnas_precision, formatos_fecha):
    if columna in columnas_precision:
        for formato in formatos_fecha:
            try:
                df[columna] = pd.to_datetime(df[columna], format=formato, errors='coerce')
                break
            except ValueError:
                continue
        return df[columna].isna().sum()
    else:
        return 0

# Función para contar los registros que no cumplen la consistencia
def contar_consistencia(df, columna, patron):
    return (~df[columna].astype(str).str.match(patron, na=False)).sum()

def contar_integridad(df, columna,columnas_integridad):
    if columna in columnas_integridad:
        condicion = (df[columna].astype(float) < 0) 
        return condicion.sum()
    else:
        condicion = ((df[columna].astype(str) == ' ') | (df[columna].astype(str) == '.') | (df[columna].astype(str) == '-1') | (df[columna].astype(str) == 'No definido') | (df[columna].astype(str) == 'Sin definir') | (df[columna].astype(str) == 'N/A') | (df[columna].astype(str) == 'Sin Clasificar') | (df[columna].astype(str) == 'Sin información'))
        return condicion.sum()

# Función para contar los registros que no cumplen la validez
def contar_validez(df, columna):
    return df[columna].isnull().sum()
    
# Función para contar los registros que no cumplen la unicidad
def contar_unicidad(df, columna,columnas_unicidad):
    if columna in columnas_unicidad:
        return df[columna].duplicated().sum()
    else:
        return 0

       
# Función para aplicar las verificaciones y generar archivos de salida
def aplicar_verificaciones(df, sheet_name, columnas, patron, archivo_salida_conteo, columnas_unicidad, columnas_precision,conteo_detallle_consolidado,formatos_fecha,columnas_integridad):
    conteo_invalido = []
    conteo_registros, num_columnas = df.shape
            
    for columna in columnas:
        conteo_precision = contar_precision(df, columna, columnas_precision,formatos_fecha)
        conteo_consistencia = contar_consistencia(df, columna, patron)
        conteo_integridad = contar_integridad(df, columna,columnas_integridad)
        conteo_validez = contar_validez(df, columna)
        conteo_unicidad = contar_unicidad(df, columna,columnas_unicidad)
        total_con_datos = df[columna].notnull().sum()     
        ValorMax = df[columna].astype(str).max()
        ValorMin = df[columna].astype(str).min()
        consistencia_dq = 100 * (1 - conteo_consistencia / conteo_registros) if conteo_registros > 0 else 0
        validez_dq = 100 * (1 - conteo_validez / conteo_registros) if conteo_registros > 0 else 0
        precisión_dq = 100 * (1 - conteo_precision / conteo_registros) if conteo_registros > 0 else 0
        integridad_dq = 100 * (1 - conteo_integridad / conteo_registros) if conteo_registros > 0 else 0       
        calidad_total = (consistencia_dq + validez_dq + precisión_dq + integridad_dq) / 4

              
        conteo_invalido.append({
            'Llave': sheet_name+'_'+columna,
            'Fuente de Información': sheet_name,
            'Campo': columna,
            'Total con Datos': total_con_datos,
            'Valor Máx': ValorMax,
            'Valor Min': ValorMin,
            'Precisión': conteo_precision,
            'Consistencia': conteo_consistencia,
            'Integridad': conteo_integridad,
            'Validez': conteo_validez,
            'Unicidad': conteo_unicidad
        })

         #Se agregó esto
        conteo_detallle_consolidado.append({
            'Llave': sheet_name+'_'+columna,
            'Fuente de Información': sheet_name,
            'Campo': columna,
            'Total con Datos': total_con_datos,
            'Total de registros': conteo_registros,
            'Valor Máx': ValorMax,
            'Valor Min': ValorMin,
            'Precisión': conteo_precision,
            'Consistencia': conteo_consistencia,
            'Integridad': conteo_integridad,
            'Validez': conteo_validez,
            'Unicidad': conteo_unicidad,
            'DQ_Consistencia': consistencia_dq,
            'DQ_Validez': validez_dq,
            'DQ_Precisión': precisión_dq,
            'DQ_Integridad': integridad_dq,
            'DQ_Calidad Total': calidad_total
            
        })
    
    #Crear Data Frame con el resultado y enviarlo a Excel
    df_conteo = pd.DataFrame(conteo_invalido)   
    df_conteo.to_excel(archivo_salida_conteo, index=False)
    
# Asignar nombres de archivos de entrada y expresión regular
archivo_entrada = 'Entrada_Fuentes_de_Datos/Extraccion_Analisis_Datos_Faculty360.xlsx'
patron_validez = '^[a-zA-Z0-9áéíóúüÀÁÉÍÓÚÜñÑçã&_. -]*$' # Patrón para caracteres válidos

print(f'Inicio del proceso, espere unos minutos..\n\n')
### Se agregó esto
conteo_detallle_consolidado = []
# Procesar cada pestaña y guardar los resultados en archivos separados
with pd.ExcelFile(archivo_entrada) as excel:
    for sheet_name in excel.sheet_names:
        if  sheet_name not in('DWH - Success Factor','Profesor Escuela Top','Profesor Facultad','Profesor Grado Académico'):#'Script Lider Académico':
            continue
        
        df = pd.read_excel(archivo_entrada,sheet_name=sheet_name)
        columnas_a_verificar = list(df.columns)
        columnas_unicidad = ['Nomina','Contrato']
        columnas_precision = ['Fecha Inicio Contrato','Fecha Fin Contrato','Fecha_Acreditacion','fecha_ultima_modificacion','DateAccreditation','CreationDate']
        columnas_integridad = ['Antigüedad']
        formatos_fecha = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%Y/%m/%d', '%Y.%m.%d']
        archivo_salida_conteo = f'Resultados_Perfilamiento/Perfilamiento_{sheet_name}.xlsx'
        aplicar_verificaciones(df, sheet_name,columnas_a_verificar, patron_validez, archivo_salida_conteo, columnas_unicidad,columnas_precision,conteo_detallle_consolidado,formatos_fecha,columnas_integridad)
        #print(f'Pestaña analizada: {sheet_name}, archivo de resultados: {archivo_salida_conteo}\n')
        print(f'Pestaña analizada: {sheet_name}, archivo de resultados: ok!\n')

print(f'\n Generación de archivo consolidado...')
df_conteo_consolidado = pd.DataFrame(conteo_detallle_consolidado)   
df_conteo_consolidado.to_excel(f'Resultados_Perfilamiento/Perfilamiento_Consolidado.xlsx', index=False)
print(f'Ok!')
print('Proceso terminado satisfactoriamente.')

Inicio del proceso, espere unos minutos..


Pestaña analizada: DWH - Success Factor, archivo de resultados: ok!

Pestaña analizada: Profesor Escuela Top, archivo de resultados: ok!

Pestaña analizada: Profesor Facultad, archivo de resultados: ok!

Pestaña analizada: Profesor Grado Académico, archivo de resultados: ok!


 Generación de archivo consolidado...
Ok!
Proceso terminado satisfactoriamente.
